# Capstone, a review pipeline

**Scenario:** a discovery production is sorted into privileged, responsive and not responsive. You
have a router, workers, and a lead that writes the log. It is the obvious build.

It is measured here against one call over the whole set, and the measurement does not flatter it.

A worker that sees one document is **a second opinion from a doctor who cannot see your file**.
Confident, quick, missing the comparison that made the first call right.

## Mechanics

Three stages, each a separate request with its own bill.

| Stage | Calls | What it sees | What it costs |
|---|---|---|---|
| Router | 1 | The whole set | One prompt, one completion |
| Workers | one per document | One document | Its own instructions, repeated per worker |
| Lead | 1 | Every worker verdict | The sum of what came back |

Round trips go from one to `documents + 2`, and shared instructions are paid once per stage. Latency
stays serial across stages even when workers run together.

## The picture

![Fan out every document against one pass plus a targeted escalation](images/review-pipeline.svg)

The dotted path is what this notebook builds first. The solid path is what the measurement argues
for.

## The cost

```
pipeline usd = router call + (documents x worker call) + lead call
one pass usd = one call over the whole set
```

The pipeline is not slightly more. It is more per document, times the document count.

## The failure

Six documents and a reviewer's answer key, so both shapes are marked the same way.

In [1]:
import json
import time

from vault import Usage, get_client, load_env, model_for, summarise

load_env()
client = get_client("02-multi-agent-orchestration/03-capstone-a-review-pipeline")

DOCS = [
    ("D-1", "Email, general counsel to outside counsel: 'Our privileged assessment of the Halden indemnity is attached, do not forward.'"),
    ("D-2", "Chat export, two engineers: 'the load test numbers we sent the client were from the tuned run, not the stock one'"),
    ("D-3", "Invoice from a vendor for twelve months of storage, no narrative, standard terms."),
    ("D-4", "Board minutes noting counsel advised on the Halden termination risk before the vote."),
    ("D-5", "Marketing deck with public product claims, distributed at a trade show."),
    ("D-6", "Email from a sales lead: 'legal said we cannot say that, so let us say it verbally on the call instead'"),
]
KEY = {"D-1": "privileged", "D-2": "responsive", "D-3": "not-responsive",
       "D-4": "privileged", "D-5": "not-responsive", "D-6": "responsive"}
SET = "\n".join(f"{doc_id}: {text}" for doc_id, text in DOCS)


def graded(rows):
    """How many calls match the answer key."""
    return sum(1 for row in rows if KEY.get(row["id"]) == row["call"])

The prompts. Every lane asks for the same shape, so the builds are comparable.

In [2]:
SHAPE = '{"id":str,"call":"privileged|responsive|not-responsive","why":str} with why under 12 words'

LANES = {
    "privilege": f"You are privilege counsel. Reply JSON only: {SHAPE}.",
    "substance": f"You are a substance reviewer. Reply JSON only: {SHAPE}.",
    "routine": f"You are a first pass reviewer. Reply JSON only: {SHAPE}.",
}
TRIAGE = ('Assign every document to a review lane. Reply JSON only: '
          '{"lanes":{"D-1":"privilege|substance|routine"}} with one entry per document.')
ONE_PASS = ('You are a document review attorney. For every document reply JSON only: '
            f'{{"reviews":[{SHAPE}]}}.')
LEAD = "You are the review lead. List the privileged set and anything to escalate."
DEEP = ('You are privilege counsel. The production set is shown for context. Decide whether the '
        'named document is privileged. Reply JSON only: {"id":str,"privileged":bool,"basis":str} '
        'with basis under 15 words.')

One caller, so no request escapes the tally. It records tokens and wall clock for each one.

In [3]:
CALLS = []


def ask(system, user, max_tokens, want_json=True):
    """Every request in this notebook goes through here, and is counted."""
    started = time.perf_counter()
    reply = client.chat.completions.create(
        model=model_for("default"), max_tokens=max_tokens,
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": user}])
    CALLS.append((Usage.from_response(reply), time.perf_counter() - started))
    text = (reply.choices[0].message.content or "").strip()
    if text.startswith("```"):
        text = text.split("```")[1].removeprefix("json").strip()
    return json.loads(text) if want_json else text

The router. One call reads the whole set and puts each document in a lane.

In [4]:
def triage(state):
    """One call over the whole set. Returns a lane per document."""
    return {"lanes": ask(TRIAGE, SET, 300)["lanes"]}

Then the worker. It runs in the lane the router chose and sees one document.

In [5]:
def review_one(state):
    """One document, one lane prompt, nothing else in view."""
    doc_id, text = state["doc"]
    row = ask(LANES[state["lane"]], f"{doc_id}: {text}", 200)
    row["id"] = doc_id
    return {"rows": [row]}

The graph wires the stages together. A run enters at the router, fans out one worker per document,
and every worker feeds the lead, which is one synthesis call over the verdicts.

In [6]:
import operator
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph
from langgraph.types import Send


class Docket(TypedDict):
    docs: list
    lanes: dict
    rows: Annotated[list, operator.add]
    log: str


graph = StateGraph(Docket)
graph.add_node("triage", triage)
graph.add_node("review", review_one)
graph.add_node("lead", lambda s: {"log": ask(LEAD, json.dumps(s["rows"]), 400, want_json=False)})
graph.add_edge(START, "triage")
graph.add_conditional_edges(
    "triage", lambda s: [Send("review", {"doc": d, "lane": s["lanes"][d[0]]}) for d in s["docs"]],
    ["review"])
graph.add_edge("review", "lead")
graph.add_edge("lead", END)
pipeline = graph.compile()

Run both, marked against the same key.

In [7]:
mark = len(CALLS)
out = pipeline.invoke({"docs": DOCS, "lanes": {}, "rows": [], "log": ""})
many = CALLS[mark:]

mark = len(CALLS)
single = ask(ONE_PASS, SET, 500)["reviews"]
one = CALLS[mark:]

for label, rows, tally in (("pipeline", out["rows"], many), ("one pass", single, one)):
    money = summarise([u for u, _ in tally])
    print(f"  {label:9} {len(tally):>2} calls  {money['total_tokens']:>5} tokens  "
          f"{money['usd']:.6f} USD  {sum(t for _, t in tally):.1f}s  "
          f"correct {graded(rows)}/{len(DOCS)}")

assert graded(out["rows"]) >= graded(single), (
    f"the pipeline got {graded(out['rows'])}/{len(DOCS)} and one call got {graded(single)}/{len(DOCS)}")

  pipeline   8 calls   1124 tokens  0.000230 USD  3.9s  correct 2/6
  one pass   1 calls    375 tokens  0.000093 USD  0.9s  correct 5/6


AssertionError: the pipeline got 2/6 and one call got 5/6

## The diagnosis

The pipeline cost more, took longer and was less correct. Nothing in it was broken.

**The router was right.** The step that reads the whole set makes good calls, because it compares
documents against each other.

**The worker lost that comparison.** Each `Send` carries one document, so the worker decides alone.
An invoice on its own reads as arguably responsive. Beside a trade show deck and a privileged memo,
it is plainly not.

**Every stage repaid the shared instructions.** The table said `documents + 2` round trips, and the
tally shows it.

Isolation protects the parent from worker output. It does not protect a worker from missing input.

## The fix

Keep the fan out and spend it where it changes an answer. One pass decides the whole set. Only
privilege calls earn a second look, and that worker gets the production set as context.

In [8]:
def escalate(doc_id):
    """A second look at one document, with the whole set still in view."""
    return ask(DEEP, f"PRODUCTION SET:\n{SET}\n\nDOCUMENT UNDER REVIEW: {doc_id}", 300)

The selection rule is the label, because privilege is the call that costs money when wrong. The cap
is a budget, so a first pass that flags everything cannot become a full pipeline.

In [9]:
FAN_OUT_CAP = 2


def select(rows, cap):
    """The documents worth a second call, and how many did not fit the cap."""
    flagged = [row["id"] for row in rows if row["call"] == "privileged"]
    return flagged[:cap], max(0, len(flagged) - cap)

Everything else stands on the first pass. Run it and compare.

In [10]:
mark = len(CALLS)
flagged, deferred = select(single, FAN_OUT_CAP)
checks = {doc_id: escalate(doc_id) for doc_id in flagged}
final = [row if checks.get(row["id"], {}).get("privileged", True) else dict(row, call="responsive")
         for row in single]
hybrid = CALLS[mark:] + one

before, after = summarise([u for u, _ in many]), summarise([u for u, _ in hybrid])
print(f"before: fan out everything  {len(many):>2} calls  {before['usd']:.6f} USD  "
      f"correct {graded(out['rows'])}/{len(DOCS)}")
print(f"after : one pass plus {len(flagged)}     {len(hybrid):>2} calls  {after['usd']:.6f} USD  "
      f"correct {graded(final)}/{len(DOCS)}")
print(f"escalated {flagged}, {deferred} deferred")
print(f"basis for {flagged[0]}: {checks[flagged[0]]['basis']!r}")

before: fan out everything   8 calls  0.000230 USD  correct 2/6
after : one pass plus 2      3 calls  0.000159 USD  correct 5/6
escalated ['D-1', 'D-4'], 0 deferred
basis for D-1: 'Attorney-client communication; legal advice from general counsel to outside counsel.'


## The gate

The regression is fanning out again when someone widens the selection rule. This pins the second look
to the flagged set and the cap.

In [11]:
def test_the_second_look_is_bounded():
    """A first pass that flags everything must not become a full fan out."""
    everything = [{"id": f"X-{i}", "call": "privileged"} for i in range(9)]
    picked, deferred = select(everything, FAN_OUT_CAP)
    assert (len(picked), deferred) == (FAN_OUT_CAP, len(everything) - FAN_OUT_CAP)
    assert select([{"id": "X-1", "call": "responsive"}], FAN_OUT_CAP) == ([], 0)


test_the_second_look_is_bounded()
print(f"gate holds: at most {FAN_OUT_CAP} second calls, and none when nothing is flagged")

gate holds: at most 2 second calls, and none when nothing is flagged


Change `select` to return `flagged` instead of `flagged[:cap]` and this test fails.

### Enterprise exploration

- Six documents fit in one call. At what set size does that stop, and what would you split first?
- The escalation reads the whole set each time. What does that cost per document at production scale?
- A wrong privilege call is a waiver risk. Who signs off, and what audit trail does the log need?
- One pass is one point of failure. What is the trade off against running it twice and comparing?

### Key takeaways

- Fan out multiplies round trips and repays the shared instructions once per stage.
- A worker cut off from the comparison set answers a narrower question than you asked.
- Measure the pipeline against one call before you build the pipeline.
- Spend a second call where it changes an answer, with the context that makes it right.